# ResNet-18 — CIFAR-10 — Seçili 18 Doğrulama Koşusu

Bu notebook, tez önerisindeki “seçilmiş deneylerin ResNet-18 üzerinde tekrarı” aşamasını yürütür. Yeni bir hiperparametre taraması değildir.

Altı tekil kombinasyon, tamamlanmış ana CNN sonuçlarından ResNet çalıştırılmadan önce sabitlenmiştir. Her biri seed 101, 202 ve 303 ile tekrarlanır: **6 kombinasyon × 3 seed = 18 koşu**.

Sabit kalanlar: CIFAR-10 veri bölünmesi ve augmentation, 50 epoch, batch size 128, kayıp fonksiyonu, optimizer ayarları, scheduler/early stopping/gradyan kırpma olmaması ve testin yalnız eğitim sonunda kullanılması. Sonuçlar Google Drive'da ana deneyden ayrı tutulur.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Sabit GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests, json
from pathlib import Path

OWNER='erdemci07'
REPO='derin-ogrenme-egitim-dinamikleri'
CODE_COMMIT='9ba373b8a86dfcfd4a6e4dabaab47d5787661377'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)

r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{CODE_COMMIT}', timeout=60)
r.raise_for_status()
commit_sha=r.json()['sha']
if commit_sha != CODE_COMMIT:
    raise RuntimeError(f'Beklenen kod commit uyuşmuyor: {commit_sha} != {CODE_COMMIT}')

a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler) != 1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir))
os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:', commit_sha)


## Google Drive ve kalıcı yollar


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_CIKTI=DRIVE_PROJE/'resnet18_secilmis_dogrulama'/'cifar10'
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)

# Çalıştırıcı ana deneyle aynı paylaşımlı CIFAR-10 ve split/statistics
# dosyalarını kullanır; sonuç klasörü ise ResNet-18'e özeldir.
veri_baglantisi=repo_dir/'veriler'
if veri_baglantisi.exists() or veri_baglantisi.is_symlink():
    veri_baglantisi.unlink()
veri_baglantisi.symlink_to(DRIVE_VERI, target_is_directory=True)

for ad in ('cifar10_main_split.json', 'cifar10_main_stats.json'):
    hedef=repo_dir/'ayarlar'/ad
    kaynak=DRIVE_AYAR/f'ana_{ad.replace("_main", "")}'
    if hedef.exists() or hedef.is_symlink():
        hedef.unlink()
    hedef.symlink_to(kaynak)

os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_CIKTI)
print('Sonuç klasörü:', DRIVE_CIKTI)
print('Veri klasörü:', DRIVE_VERI)


## Bağımlılıklar ve GPU doğrulaması


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'
print('GPU:', gpu)
if not torch.cuda.is_available() or 'T4' not in gpu.upper():
    raise RuntimeError(f'ResNet-18 doğrulaması için NVIDIA T4 seçilmelidir. Mevcut: {gpu}')


## Veri kaynağını doğrula


In [ ]:
beklenen1=DRIVE_VERI/'cifar-10-batches-py'
beklenen2=DRIVE_VERI/'cifar-10-python.tar.gz'
if not beklenen1.is_dir() and not beklenen2.is_file():
    raise FileNotFoundError(f'CIFAR-10 tez_datasets içinde bulunamadı: {beklenen1} veya {beklenen2}')
print('CIFAR-10 kaynağı hazır.')


## Ön kontrol

Bu hücre gerçek eğitim başlamadan CIFAR-10 girdisini, ResNet-18'deki üç aktivasyonu, altı optimizer–aktivasyon eşleşmesini ve kayıp fonksiyonunu tek batch üzerinde doğrular. “ÖN KONTROL BAŞARILI” çıktısı olmadan koşulara geçilmez.


In [ ]:
import importlib
import calistir_resnet18_dogrulama as resnet
importlib.reload(resnet)
resnet.ana(['--sadece-on-kontrol'])


## 18 koşuyu çalıştır / devam ettir

Runtime kesilirse notebook tekrar çalıştırılabilir. Tamamlanan koşular atlanır; yarım koşu checkpoint'ten sürdürülür. Ana CNN sonuçlarıyla karışmaması için çıktılar ayrı ResNet-18 klasörüne yazılır.


In [ ]:
importlib.reload(resnet)
resnet.ana([])


## İlerleme özeti


In [ ]:
tamam={}
kararsiz={}
for p in DRIVE_CIKTI.glob('*/tamamlandi.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
        tamam[d['run_id']]=d
    except Exception:
        pass
for p in DRIVE_CIKTI.glob('*/kararsiz.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
        if d['run_id'] not in tamam:
            kararsiz[d['run_id']]=d
    except Exception:
        pass
deger=len(tamam)+len(kararsiz)
print(f'Değerlendirilen: {deger}/18 | tamam={len(tamam)} | kararsız={len(kararsiz)}')
if deger < 18:
    print('Kesinti olursa aynı notebooku yeniden Run all yapın.')
else:
    print('Seçili ResNet-18 doğrulama koşuları tamamlandı/değerlendirildi.')
